# Remaining paper runs

Run **1. Setup**, then **2. Check what is left**. Click the individual ▶ buttons. The notebook opens with **TRAINING_MODE = False**. Change it to **True** and rerun setup when you want to start jobs. Keep the default tags to reuse saved runs. The GPU cells run one job at a time and save logs and utilization on D:.

[Paper training checklist](../../../PAPER_TRAINING_CHECKLIST.md) · [Reviewer checklist](../../../REVIEWER_CHECKLIST.md)

Rerun Setup after helper updates to load the latest code.

## 1. Setup

In [1]:
from pathlib import Path
import importlib
import sys

if Path(sys.prefix).name.casefold() != "afterimage_env" or sys.version_info[:2] != (3, 11):
    raise RuntimeError(f"Select the Python 3.11 afterimage_env kernel. Current: {sys.executable}")

sys.dont_write_bytecode = True
QWEN_DIR = Path(r"D:\Afterimage_Reproduction\finetuning\Qwen3")
HELPER_DIR = QWEN_DIR / "paper_revision"  # pinned paper_reproduction.py stays in QWEN_DIR
for directory in (QWEN_DIR, HELPER_DIR):
    if str(directory) in sys.path:
        sys.path.remove(str(directory))
    sys.path.insert(0, str(directory))
importlib.invalidate_caches()

def load_helper(name):
    module = importlib.reload(sys.modules[name]) if name in sys.modules else importlib.import_module(name)
    if Path(module.__file__).resolve() != (HELPER_DIR / f"{name}.py").resolve():
        raise ImportError(f"Expected {name} from {HELPER_DIR}, got {module.__file__}")
    return module

reproduction = load_helper("reproduce_run_02")
remaining = load_helper("remaining_runs")

TRAINING_MODE = False  # Change to True and rerun this cell to enable GPU jobs.
ORIGINAL_RUN_TAG = "run02_reproduction_01"
BUILD_TAG = "aptos_original_split_01"
QWEN_RUN_TAG = "01"
CONTROL_TAG = "paper_controls_01"

BF16_BASE = Path(r"D:\Afterimage_Experiments\assets\qwen3-vl-8b-bf16")

MESSIDOR_TAG = "verified_messidor_01"
MESSIDOR_ROOT = Path(r"D:\Afterimage_Experiments\datasets\messidor_verified")
MESSIDOR = {
    "512_on": MESSIDOR_ROOT / "clahe_on_512.csv",
    "512_off": MESSIDOR_ROOT / "clahe_off_512.csv",
    "224_off": MESSIDOR_ROOT / "standard_224.csv",
}

def launch(job, *args, **kwargs):
    if not TRAINING_MODE:
        print("Set TRAINING_MODE = True and rerun Setup, then run this cell.")
        return
    return job(*args, **kwargs)


## 2. Check what is left

This cell only reads saved results.

In [2]:
remaining.show_checklist(QWEN_RUN_TAG, ORIGINAL_RUN_TAG, CONTROL_TAG)

Pair 1 CLAHE-on, BF16 scoring: pending
r8_off duplicate_excluded, seed pair 1: running (last saved marker; check live logs)
r8_off duplicate_excluded, seed pair 2: pending
r8_off duplicate_excluded, seed pair 3: pending
r32_off duplicate_excluded, seed pair 1: pending
r32_off duplicate_excluded, seed pair 2: pending
r32_off duplicate_excluded, seed pair 3: pending
r16_on previous_split, seed pair 1: pending
r16_on previous_split, seed pair 2: pending
r16_on previous_split, seed pair 3: pending
BF16 base, seed pair 1: pending
RetiZero, seed 42: pending
Deferred historical repeat 1: pending
BF16 base, seed pair 2: pending
RetiZero, seed 43: pending
Deferred historical repeat 2: pending
BF16 base, seed pair 3: pending
RetiZero, seed 44: pending
Deferred historical repeat 3: pending


## 3. Finish Pair 1 scoring (no retraining)

Score the saved CLAHE-on adapter in BF16 on the **367 original APTOS test photos**. The other five scores already use BF16. Use the next cell to compare the three pairs after scoring.

In [3]:
launch(remaining.score_clahe_on_pair1, ORIGINAL_RUN_TAG, BUILD_TAG)

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [4]:
reproduction.summarize(ORIGINAL_RUN_TAG)

clahe_on 2 {'accuracy': 0.49319, 'qwk': 0.0, 'macro_ovr_auc': 0.48697, 'grade_recall': [1.0, 0.0, 0.0, 0.0, 0.0]}
clahe_on 3 {'accuracy': 0.62943, 'qwk': 0.7775, 'macro_ovr_auc': 0.89359, 'grade_recall': [0.9834, 0.4324, 0.22, 0.7368, 0.0333]}
clahe_on 2/3 complete {'accuracy': {'mean': 0.5613079019073569, 'range': [0.49318801089918257, 0.6294277929155313], 'sample_sd': 0.09633607373113723}, 'qwk': {'mean': 0.38875060379925275, 'range': [0.0, 0.7775012075985055], 'sample_sd': 0.5497763762736328}, 'macro_ovr_auc': {'mean': 0.6902836339260937, 'range': [0.4869735310292084, 0.893593736822979], 'sample_sd': 0.28752390488424473}}
clahe_off 1 {'accuracy': 0.77929, 'qwk': 0.8979, 'macro_ovr_auc': 0.93574, 'grade_recall': [0.9116, 0.5135, 0.79, 0.2632, 0.6]}
clahe_off 2 {'accuracy': 0.6485, 'qwk': 0.827, 'macro_ovr_auc': 0.90279, 'grade_recall': [0.7017, 0.7297, 0.67, 0.4737, 0.2667]}
clahe_off 3 {'accuracy': 0.7248, 'qwk': 0.87307, 'macro_ovr_auc': 0.93481, 'grade_recall': [0.8895, 0.7568, 0.

## 4. Finish rank 8 and rank 32 CLAHE-off

These six runs finish the duplicate-excluded rank × CLAHE study on the **361-photo APTOS test set**. Keep QWEN_RUN_TAG = "01" for the interrupted rank-8 seed-pair-1 run. Rerun the same cell with the same tag to resume or reuse a completed result.

In [5]:
# Rank 8, CLAHE off, seed pair 1
launch(remaining.qwen_remaining, "r8_off", 1, "duplicate_excluded", QWEN_RUN_TAG)

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [6]:
# Rank 32, CLAHE off, seed pair 1
launch(remaining.qwen_remaining, "r32_off", 1, "duplicate_excluded", QWEN_RUN_TAG)

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [7]:
# Rank 8, CLAHE off, seed pair 2
launch(remaining.qwen_remaining, "r8_off", 2, "duplicate_excluded", QWEN_RUN_TAG)

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [8]:
# Rank 32, CLAHE off, seed pair 2
launch(remaining.qwen_remaining, "r32_off", 2, "duplicate_excluded", QWEN_RUN_TAG)

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [9]:
# Rank 8, CLAHE off, seed pair 3
launch(remaining.qwen_remaining, "r8_off", 3, "duplicate_excluded", QWEN_RUN_TAG)

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [10]:
# Rank 32, CLAHE off, seed pair 3
launch(remaining.qwen_remaining, "r32_off", 3, "duplicate_excluded", QWEN_RUN_TAG)

Set TRAINING_MODE = True and rerun Setup, then run this cell.


## 5. Compare duplicate exclusion

Run these three controls if the paper will report the effect of removing exact duplicates. They restore the previous train/validation split and still score the same 361 test photos as the cleaned runs.

In [11]:
# Rank 16, CLAHE on, previous train/validation split, seed pair 1
launch(remaining.qwen_remaining, "r16_on", 1, "previous_split", QWEN_RUN_TAG)

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [12]:
# Rank 16, CLAHE on, previous train/validation split, seed pair 2
launch(remaining.qwen_remaining, "r16_on", 2, "previous_split", QWEN_RUN_TAG)

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [13]:
# Rank 16, CLAHE on, previous train/validation split, seed pair 3
launch(remaining.qwen_remaining, "r16_on", 3, "previous_split", QWEN_RUN_TAG)

Set TRAINING_MODE = True and rerun Setup, then run this cell.


## 6. QLoRA comparison (BF16 base + LoRA)

Reviewer S7. These full runs use the unquantized BF16 base and keep 16 frozen decoder layers in system RAM. Run one seed at a time. The same cell resumes its saved checkpoint or retries evaluation. The existing 4-bit rank-16 CLAHE-on runs are the reference.

In [14]:
remaining.qwen_bf16("check", 1, QWEN_RUN_TAG, BF16_BASE, control_tag=CONTROL_TAG, offload_layers=16)

Log: D:\Afterimage_Experiments\logs\training\full_bf16_r16_on_duplicate_excluded_t42_a3407_paper_controls_01\check.log
GPU 0: NVIDIA GeForce RTX 5070 Ti, 16303 MiB total / 14630 MiB free
Use one visible CUDA GPU with room for the 16.4 GiB BF16 base plus activations. Start with 24 GiB or more and confirm capacity in the matched run.
Point --base-snapshot at the local unquantized BF16 Qwen3-VL-8B-Instruct weights.


RuntimeError: Worker exited with code 1. See D:\Afterimage_Experiments\logs\training\full_bf16_r16_on_duplicate_excluded_t42_a3407_paper_controls_01\check.log.

### Optional 16 GB CPU-offload pilot

The value of offload_layers sets how many frozen decoder layers use system RAM. This pilot runs two LoRA updates and a small validation pass and writes its own logs under D:/Afterimage_Experiments/checks.

In [ ]:
launch(remaining.bf16_shared_memory_pilot, BF16_BASE, QWEN_RUN_TAG, offload_layers=16)

In [ ]:
# BF16 base, rank 16, CLAHE on, seed pair 1
launch(remaining.qwen_bf16, "run", 1, QWEN_RUN_TAG, BF16_BASE, control_tag=CONTROL_TAG, offload_layers=16)

In [ ]:
# BF16 base, rank 16, CLAHE on, seed pair 2
launch(remaining.qwen_bf16, "run", 2, QWEN_RUN_TAG, BF16_BASE, control_tag=CONTROL_TAG, offload_layers=16)

In [ ]:
# BF16 base, rank 16, CLAHE on, seed pair 3
launch(remaining.qwen_bf16, "run", 3, QWEN_RUN_TAG, BF16_BASE, control_tag=CONTROL_TAG, offload_layers=16)

## 7. RetiZero: moved

RetiZero now has its own folder and notebook: [`finetuning/RetiZero/notebooks/02_retizero_runs.ipynb`](../../RetiZero/notebooks/02_retizero_runs.ipynb). It covers the replay of the saved checkpoint, the three seeds on the same 361 test IDs as Qwen, the repeats on the original split, timing, and Messidor scoring, with a pilot and a queue like `03_final_runs.ipynb`. The pretrained checkpoint path and its SHA-256 are set in that notebook's Setup.

## 8. Score verified Messidor (evaluation only)

Use the team's raw photos, independent grades and eye pairing to prepare the three CSVs in Setup. The source notebook names **IMAGES.zip.001 and its continuation parts**, **messidor_data.csv**, and **messidor-2.csv**. Each CSV uses `id_code,image_path,label,source_dataset,split,patient_id`, with absolute PNG paths, `messidor` as the source, and `test` as the split. Match image IDs, grades and pairing IDs across 512-pixel CLAHE-on/off and 224-pixel standard inputs. The archived replay already exists and has its own saved results. Reconcile the four identical photo pairs assigned to different patient IDs in that archive before choosing the final cohort.RetiZero is scored on Messidor from its own notebook (see section 7).

In [ ]:
remaining.check_messidor(MESSIDOR)

In [ ]:
# Select one saved Qwen experiment and run this cell for each model needed in the paper.
# Series choices: original, duplicate_excluded, previous_split, bf16_base.
MESSIDOR_SERIES = "original"
MESSIDOR_CONDITION = "r16_on"
MESSIDOR_INDEX = 1
launch(remaining.score_messidor_qwen, MESSIDOR_SERIES, MESSIDOR_CONDITION,
       MESSIDOR_INDEX, MESSIDOR, QWEN_RUN_TAG, ORIGINAL_RUN_TAG, CONTROL_TAG, MESSIDOR_TAG)

## 9. Deferred historical training repeats

Use these only if the paper will claim a fresh reproduction of historical training. All three use the original archived CLAHE-on data and fixed seeds. The saved-adapter replay is already complete.

In [ ]:
# Historical repeat 1
launch(reproduction.train_repeat, ORIGINAL_RUN_TAG, 1)

In [ ]:
# Historical repeat 2
launch(reproduction.train_repeat, ORIGINAL_RUN_TAG, 2)

In [ ]:
# Historical repeat 3
launch(reproduction.train_repeat, ORIGINAL_RUN_TAG, 3)

## 10. Recheck progress

The training checklist records the experiments. The reviewer checklist also covers methods, tuning justification, augmentation counts, and error analysis.

In [ ]:
remaining.show_checklist(QWEN_RUN_TAG, ORIGINAL_RUN_TAG, CONTROL_TAG)